# Создание рекомендаций 3 - х видов


Загружаем библиотеки необходимые для выполнения кода ноутбука.

In [39]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import boto3
from dotenv import load_dotenv
import sys
import scipy.sparse
import sklearn
from utils import check_numeric_dtypes, compare_csv_parquet
from IPython.display import display, HTML
from sklearn.preprocessing import LabelEncoder
import gc
import numpy as np
import pandas as pd
import scipy.sparse
import pyarrow as pa
import pyarrow.parquet as pq
from implicit.als import AlternatingLeastSquares


load_dotenv()

events = pd.read_parquet(
    "parquet_cleaned/events.parquet"
)

item_properties = pd.read_parquet(
    "parquet_cleaned/item_properties.parquet"
)

In [40]:
import boto3, os
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)
bucket_name = os.getenv("S3_BUCKET_NAME")

In [41]:
# s3.upload_file(
#     "parquet_cleaned/item_properties.parquet",
#     bucket_name,
#     "recsys/data/items.parquet"
# )

# s3.upload_file(
#     "parquet_cleaned/events.parquet",
#     bucket_name,
#     "recsys/data/events.parquet"
# )

# Разбиение данных

Разбиваем данные на тренировочную, тестовую выборки.

# Топ популярных

Рассчитаем рекомендации как топ популярных.

In [42]:
# Конец доступных данных
data_end = (
    events["timestamp"].max()
    + pd.Timedelta(milliseconds=1)
)

# Последние 7 дней — финальная evaluation
eval_start = (
    data_end
    - pd.Timedelta(days=7)
)

# Предыдущие 7 дней — обучение ranker
ranker_start = (
    eval_start
    - pd.Timedelta(days=7)
)

print("Ranker start:", ranker_start)
print("Eval start:", eval_start)
print("Data end:", data_end)

Ranker start: 2015-09-04 02:59:47.789000
Eval start: 2015-09-11 02:59:47.789000
Data end: 2015-09-18 02:59:47.789000


In [43]:
# История для обучения ALS / Top Popular / Similar
train = events[
    events["timestamp"] < ranker_start
].copy()

# Окно, на котором потом обучается ranker
test = events[
    (events["timestamp"] >= ranker_start)
    & (events["timestamp"] < eval_start)
].copy()

In [44]:
events.head(2)

,timestamp,visitorid,event,itemid,transactionid
0,2015-06-02 05:02:12.117,257597,view,355908,<NA>
1,2015-06-02 05:50:14.164,992329,view,248676,<NA>


In [45]:
TOP_N = 100

positive_events = (
    train[
        train["event"].isin(
            ["addtocart", "transaction"]
        )
    ]
    [["visitorid", "itemid"]]
    .drop_duplicates()
)
positive_events.head(2)

,visitorid,itemid
17,287857,5206
19,158090,10572


In [46]:

top_popular = (
    positive_events
    .groupby("itemid", as_index=False)
    .size()
    .rename(columns={"size": "score"})
    .sort_values(
        "score",
        ascending=False
    )
    .head(TOP_N)
    .reset_index(drop=True)
)



In [47]:
top_popular.head(2)

,itemid,score
0,461686,242
1,312728,137


In [48]:
top_popular["rank"] = range(
    1,
    len(top_popular) + 1
)

top_popular.head(2)

,itemid,score,rank
0,461686,242,1
1,312728,137,2


In [49]:
import os

# папка для рекомендаций топ популярных товаров
TOP_POPULAR_DIR = "recommendations/top_popular"

os.makedirs(
    TOP_POPULAR_DIR,
    exist_ok=True
)

top_popular.to_parquet(
    f"{TOP_POPULAR_DIR}/top_popular.parquet",
    index=False
)

In [50]:
from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

user_encoder.fit(train["visitorid"])
item_encoder.fit(train["itemid"])

train["user_idx"] = user_encoder.transform(
    train["visitorid"]
)

train["item_idx"] = item_encoder.transform(
    train["itemid"]
)
train.head(2)

,timestamp,visitorid,event,itemid,transactionid,user_idx,item_idx
0,2015-06-02 05:02:12.117,257597,view,355908,<NA>,234166,172014
1,2015-06-02 05:50:14.164,992329,view,248676,<NA>,902572,120202


In [51]:
test_known = test[
    test["visitorid"].isin(user_encoder.classes_)
    & test["itemid"].isin(item_encoder.classes_)
].copy()

test_known.head(2)

,timestamp,visitorid,event,itemid,transactionid
1200043,2015-09-04 04:38:18.537,401923,view,142219,<NA>
1200050,2015-09-04 05:43:51.339,709702,view,426031,<NA>


In [52]:
test_known["user_idx"] = user_encoder.transform(
    test_known["visitorid"]
)

test_known["item_idx"] = item_encoder.transform(
    test_known["itemid"]
)
test_known.head(2)

,timestamp,visitorid,event,itemid,transactionid,user_idx,item_idx
1200043,2015-09-04 04:38:18.537,401923,view,142219,<NA>,365549,68947
1200050,2015-09-04 05:43:51.339,709702,view,426031,<NA>,645546,205788


In [53]:
train_als = train[
    [
        "user_idx",
        "item_idx",
        "event"
    ]
].copy()

In [54]:
user_events = train.groupby("visitorid")["event"].apply(set)

def conversion_to(target, source):
    has_s = user_events.apply(lambda s: source in s)
    has_t = user_events.apply(lambda s: target in s)
    return (has_s & has_t).sum() / max(has_s.sum(), 1)

p_cart_given_view = max(conversion_to("addtocart", "view"), 1e-3)

event_weights = {
    "view":        1.0,
    "addtocart":   float(np.sqrt(1.0 / p_cart_given_view)),        # ~6.4
    "transaction": float(np.sqrt(1.0 / p_cart_given_view) * 1.3),  # ~8.3
}

In [55]:
event_weights

{'view': 1.0, 'addtocart': 6.374881508723317, 'transaction': 8.287345961340312}

In [56]:


train_als["weight"] = (
    train_als["event"]
    .map(event_weights)
    .astype("float32")
)

In [57]:
train_als.drop(
    columns=["event"],
    inplace=True
)

In [58]:
train_als.head(10)

,user_idx,item_idx,weight
0,234166,172014,1.0
1,902572,120202,1.0
2,100873,154084,1.0
3,439927,122408,1.0
4,865314,177598,1.0
5,884700,10814,1.0
6,737452,213985,1.0
7,722397,212126,1.0
8,750388,207107,1.0
9,308689,39997,1.0


In [59]:
from scipy.sparse import csr_matrix

user_item_matrix = csr_matrix(
    (
        train_als["weight"].astype("float32"),
        (
            train_als["user_idx"].astype("int32"),
            train_als["item_idx"].astype("int32")
        )
    ),
    shape=(
        len(user_encoder.classes_),
        len(item_encoder.classes_)
    )
)



In [60]:
print("Shape:", user_item_matrix.shape)
print("Interactions:", user_item_matrix.nnz)
print("Users:", len(user_encoder.classes_))
print("Items:", len(item_encoder.classes_))

Shape: (1280329, 225427)
Interactions: 1950423
Users: 1280329
Items: 225427


In [61]:
from implicit.als import AlternatingLeastSquares


model = AlternatingLeastSquares(
    factors=32,              # 32–128
    regularization=0.05,     # 0.01–0.1
    alpha=10,               # 0.1–2 (НЕ 40!)
    iterations=15,           # 10–20
    random_state=42,
    calculate_training_loss=True,  # для мониторинга
)

model.fit(user_item_matrix)

  0%|          | 0/15 [00:00<?, ?it/s]

In [62]:
user_idx = 100

item_ids, scores = model.recommend(
    userid=user_idx,
    user_items=user_item_matrix[user_idx],
    N=10,
    filter_already_liked_items=False
)

In [63]:
recommended_itemids = item_encoder.inverse_transform(
    item_ids
)

In [64]:
recommendations = pd.DataFrame({
    "item_idx": item_ids,
    "itemid": recommended_itemids,
    "score": scores
})



In [65]:
recommendations["rank"] = range(
    1,
    len(recommendations) + 1
)

recommendations

,item_idx,itemid,score,rank
0,131686,272455,0.001747,1
1,3815,7943,0.001725,2
2,110110,227880,0.001360,3
3,128509,265810,0.001291,4
4,158967,329279,0.001284,5
5,28515,58752,0.001218,6
6,143771,297513,0.001127,7
7,39110,80554,0.001126,8
8,180471,373600,0.001109,9
9,37144,76617,0.001104,10


In [66]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# --------------------------------------------------
# Параметры
# --------------------------------------------------

TOP_N = 100
BATCH_SIZE = 100000

output_dir = "recommendations/personal_als"
output_path = f"{output_dir}/personal_als.parquet"

os.makedirs(
    output_dir,
    exist_ok=True
)

# --------------------------------------------------
# Пользователи, для которых строим рекомендации
# --------------------------------------------------

users_for_recommendations = np.arange(
    user_item_matrix.shape[0],
    dtype=np.int32
)

total_users = len(users_for_recommendations)

print(
    f"Пользователей для рекомендаций: "
    f"{total_users:,}"
)

# --------------------------------------------------
# Генерация рекомендаций батчами
# --------------------------------------------------

writer = None

try:
    for start in range(
        0,
        total_users,
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            total_users
        )

        batch_users = (
            users_for_recommendations[start:end]
        )

        # Получаем персональные рекомендации
        item_ids_enc, scores = model.recommend(
            userid=batch_users,
            user_items=user_item_matrix[
                batch_users
            ],
            N=TOP_N,
            filter_already_liked_items=False
        )

        batch_size = len(batch_users)

        # Возвращаем реальные visitorid
        original_user_ids = (
            user_encoder.classes_[
                batch_users
            ]
        )

        # Возвращаем реальные itemid
        original_item_ids = (
            item_encoder.classes_[
                item_ids_enc.reshape(-1)
            ]
        )

        # Формируем batch
        personal_batch = pd.DataFrame({
            "visitorid": np.repeat(
                original_user_ids,
                TOP_N
            ),

            "itemid": original_item_ids,

            "score": (
                scores
                .reshape(-1)
                .astype(np.float32)
            ),

            "rank": np.tile(
                np.arange(
                    1,
                    TOP_N + 1,
                    dtype=np.int16
                ),
                batch_size
            )
        })

        # Переводим batch в Arrow Table
        table = pa.Table.from_pandas(
            personal_batch,
            preserve_index=False
        )

        # Создаём writer один раз
        if writer is None:
            writer = pq.ParquetWriter(
                output_path,
                table.schema,
                compression="snappy"
            )

        # Дописываем очередной batch
        writer.write_table(table)

        # Освобождаем память
        del personal_batch
        del table
        del item_ids_enc
        del scores
        del original_user_ids
        del original_item_ids

        # Прогресс
        
        print(
                f"Обработано: "
                f"{end:,} / {total_users:,} "
                f"({end / total_users:.1%})"
            )

finally:
    if writer is not None:
        writer.close()

print(
    f"Рекомендации сохранены: "
    f"{output_path}"
)

Пользователей для рекомендаций: 1,280,329
Обработано: 100,000 / 1,280,329 (7.8%)
Обработано: 200,000 / 1,280,329 (15.6%)
Обработано: 300,000 / 1,280,329 (23.4%)
Обработано: 400,000 / 1,280,329 (31.2%)
Обработано: 500,000 / 1,280,329 (39.1%)
Обработано: 600,000 / 1,280,329 (46.9%)
Обработано: 700,000 / 1,280,329 (54.7%)
Обработано: 800,000 / 1,280,329 (62.5%)
Обработано: 900,000 / 1,280,329 (70.3%)
Обработано: 1,000,000 / 1,280,329 (78.1%)
Обработано: 1,100,000 / 1,280,329 (85.9%)
Обработано: 1,200,000 / 1,280,329 (93.7%)
Обработано: 1,280,329 / 1,280,329 (100.0%)
Рекомендации сохранены: recommendations/personal_als/personal_als.parquet


In [67]:
# # ------------------------------------------------------------
# # Загружаем personal_als.parquet в S3
# # ------------------------------------------------------------

# s3.upload_file(
#     "recommendations/personal_als/personal_als.parquet",
#     bucket_name,
#     "recsys/recommendations/personal_als.parquet"
# )

# print(
#     "Файл personal_als.parquet успешно загружен "
#     "в recsys/recommendations/"
# )

In [68]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(
    "recommendations/personal_als/personal_als.parquet"
)

print(
    "Количество строк:",
    parquet_file.metadata.num_rows
)

print(
    "Количество row groups:",
    parquet_file.metadata.num_row_groups
)

first_part = (
    parquet_file
    .read_row_group(0)
    .to_pandas()
)

first_part.head(20)

Количество строк: 128032900
Количество row groups: 128


,visitorid,itemid,score,rank
0,1,219512,0.005724,1
1,1,384302,0.005392,2
2,1,161623,0.004245,3
3,1,190000,0.004224,4
4,1,393028,0.003915,5
5,1,91105,0.003414,6
6,1,206880,0.003230,7
7,1,274798,0.003077,8
8,1,289103,0.003066,9
9,1,11279,0.003020,10


In [69]:
# ------------------------------------------------------------
# Проверяем похожие товары для одного item
# ------------------------------------------------------------

item_id_enc = 0

similar_items_enc, scores = model.similar_items(
    itemid=item_id_enc,
    N=11
)

print("Исходный item_id_enc:", item_id_enc)
print("Похожие item_id_enc:", similar_items_enc)
print("Scores:", scores)

Исходный item_id_enc: 0
Похожие item_id_enc: [     0  53961 207341  42204 130614 162736 192725 172750 162357  29713
 170671]
Scores: [1.         0.9999999  0.9993396  0.9983177  0.9982989  0.9980129
 0.9980009  0.98965394 0.98936117 0.98679024 0.98679006]


In [70]:
original_item_id = item_encoder.classes_[
    item_id_enc
]

similar_original_item_ids = item_encoder.classes_[
    similar_items_enc
]

similar_items = pd.DataFrame({
    "item_idx": similar_items_enc,
    "itemid": similar_original_item_ids,
    "score": scores
})

In [71]:
similar_items = similar_items[
    similar_items["item_idx"] != item_id_enc
].head(10)

similar_items["rank"] = range(
    1,
    len(similar_items) + 1
)

similar_items

,item_idx,itemid,score,rank
1,53961,111300,1.000000,1
2,207341,429337,0.999340,2
3,42204,86962,0.998318,3
4,130614,270194,0.998299,4
5,162736,336863,0.998013,5
6,192725,398963,0.998001,6
7,172750,357397,0.989654,7
8,162357,336073,0.989361,8
9,29713,61208,0.986790,9
10,170671,353107,0.986790,10


In [72]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# --------------------------------------------------
# Параметры
# --------------------------------------------------

SIMILAR_N = 50
BATCH_SIZE = 20000

output_dir = "recommendations/similar"
output_path = f"{output_dir}/similar.parquet"

os.makedirs(
    output_dir,
    exist_ok=True
)

# --------------------------------------------------
# Все товары, которые знает ALS
# --------------------------------------------------

items_for_similarity = np.arange(
    len(item_encoder.classes_),
    dtype=np.int32
)

total_items = len(items_for_similarity)

print(
    f"Всего товаров: {total_items:,}"
)

writer = None

try:

    for start in range(
        0,
        total_items,
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            total_items
        )

        batch_items = items_for_similarity[
            start:end
        ]

        # --------------------------------------------------
        # Получаем похожие товары
        #
        # N + 1, потому что первым результатом обычно
        # является сам исходный товар
        # --------------------------------------------------

        similar_ids, scores = model.similar_items(
            itemid=batch_items,
            N=SIMILAR_N + 1
        )

        source_ids = []
        target_ids = []
        batch_scores = []
        ranks = []

        # --------------------------------------------------
        # Обрабатываем результаты каждого товара
        # --------------------------------------------------

        for row_idx, source_item_idx in enumerate(
            batch_items
        ):

            row_ids = similar_ids[row_idx]
            row_scores = scores[row_idx]

            # исключаем сам товар
            mask = row_ids != source_item_idx

            row_ids = row_ids[mask][
                :SIMILAR_N
            ]

            row_scores = row_scores[mask][
                :SIMILAR_N
            ]

            count = len(row_ids)

            source_ids.extend(
                [source_item_idx] * count
            )

            target_ids.extend(
                row_ids
            )

            batch_scores.extend(
                row_scores
            )

            ranks.extend(
                range(
                    1,
                    count + 1
                )
            )

        # --------------------------------------------------
        # Encoded ID -> исходные itemid
        # --------------------------------------------------

        source_ids = np.asarray(
            source_ids,
            dtype=np.int32
        )

        target_ids = np.asarray(
            target_ids,
            dtype=np.int32
        )

        similar_batch = pd.DataFrame({

            "itemid": item_encoder.classes_[
                source_ids
            ],

            "similar_itemid": item_encoder.classes_[
                target_ids
            ],

            "score": np.asarray(
                batch_scores,
                dtype=np.float32
            ),

            "rank": np.asarray(
                ranks,
                dtype=np.int16
            )
        })

        # --------------------------------------------------
        # Записываем batch в parquet
        # --------------------------------------------------

        table = pa.Table.from_pandas(
            similar_batch,
            preserve_index=False
        )

        if writer is None:

            writer = pq.ParquetWriter(
                output_path,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)

        # --------------------------------------------------
        # Освобождаем память
        # --------------------------------------------------

        del similar_batch
        del table
        del similar_ids
        del scores
        del source_ids
        del target_ids
        del batch_scores
        del ranks

        # --------------------------------------------------
        # Прогресс
        # --------------------------------------------------

        

        print(
                f"Обработано: "
                f"{end:,} / {total_items:,} "
                f"({end / total_items:.1%})"
            )

finally:

    if writer is not None:
        writer.close()


print(
    f"Похожие товары сохранены: "
    f"{output_path}"
)

Всего товаров: 225,427
Обработано: 20,000 / 225,427 (8.9%)
Обработано: 40,000 / 225,427 (17.7%)
Обработано: 60,000 / 225,427 (26.6%)
Обработано: 80,000 / 225,427 (35.5%)
Обработано: 100,000 / 225,427 (44.4%)
Обработано: 120,000 / 225,427 (53.2%)
Обработано: 140,000 / 225,427 (62.1%)
Обработано: 160,000 / 225,427 (71.0%)
Обработано: 180,000 / 225,427 (79.8%)
Обработано: 200,000 / 225,427 (88.7%)
Обработано: 220,000 / 225,427 (97.6%)
Обработано: 225,427 / 225,427 (100.0%)
Похожие товары сохранены: recommendations/similar/similar.parquet


In [73]:
# s3.upload_file(
#     "recommendations/similar/similar.parquet",
#     bucket_name,
#     "recsys/recommendations/similar.parquet"
# )

# print(
#     "Файл similar.parquet успешно загружен "
#     "в recsys/recommendations/"
# )

In [74]:
positive_events_test = (
    test[                                          # ← test, не train
        test["event"].isin(["addtocart", "transaction"])
    ]
    [["visitorid", "itemid"]]
    .drop_duplicates()
)

positive_pairs = positive_events_test.rename(columns={
    "visitorid": "user_id",
    "itemid": "item_id"
})

train_pairs = (
    train[["visitorid", "itemid"]]
    .drop_duplicates()
)

positive_seen_in_train = positive_pairs.merge(
    train_pairs,
    left_on=["user_id", "item_id"],
    right_on=["visitorid", "itemid"],
    how="inner"
)

print(
    "Positive pairs, которые уже были в train:",
    len(positive_seen_in_train)
)

print(
    "Доля:",
    len(positive_seen_in_train) / len(positive_pairs)
)

Positive pairs, которые уже были в train: 137
Доля: 0.04384


In [75]:
positive_items_known = positive_pairs[
    positive_pairs["item_id"].isin(item_encoder.classes_)
]

print(
    "Positive pairs с известными ALS товарами:",
    len(positive_items_known)
)

print(
    "Доля:",
    len(positive_items_known) / len(positive_pairs)
)

Positive pairs с известными ALS товарами: 2964
Доля: 0.94848


In [76]:
personal_als = pd.read_parquet(
    "recommendations/personal_als/personal_als.parquet"
)

positive_in_personal_als = positive_pairs.merge(
    personal_als,
    left_on=["user_id", "item_id"],
    right_on=["visitorid", "itemid"],
    how="inner"
)

print(
    "Positive найдено в personal_als:",
    len(positive_in_personal_als)
)

print(
    "Coverage personal_als:",
    len(positive_in_personal_als) / len(positive_pairs)
)

Positive найдено в personal_als: 86
Coverage personal_als: 0.02752


In [77]:
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
import pandas as pd
import numpy as np


# ============================================================
# 1. ФИКСИРУЕМ ВЕСА СОБЫТИЙ
# ============================================================

weights = event_weights


# ============================================================
# 2. ГОТОВИМ ДАННЫЕ ДЛЯ ALS
# ============================================================

train_als_test = train[
    [
        "user_idx",
        "item_idx",
        "event"
    ]
].copy()


train_als_test["weight"] = (
    train_als_test["event"]
    .map(weights)
    .astype("float32")
)


# ============================================================
# 3. СТРОИМ USER-ITEM МАТРИЦУ
# ============================================================

user_item_matrix_test = csr_matrix(
    (
        train_als_test["weight"],
        (
            train_als_test["user_idx"].astype("int32"),
            train_als_test["item_idx"].astype("int32")
        )
    ),
    shape=(
        len(user_encoder.classes_),
        len(item_encoder.classes_)
    )
)


print(
    "Матрица:",
    user_item_matrix_test.shape
)

print(
    "Количество взаимодействий:",
    user_item_matrix_test.nnz
)


# ============================================================
# 4. ОБУЧАЕМ ALS
# ============================================================

model_test = AlternatingLeastSquares(
    factors=32,
    regularization=0.05,
    alpha=10,
    iterations=15,
    random_state=42,
    calculate_training_loss=True,
)

model_test.fit(
    user_item_matrix_test
)


# ============================================================
# 5. ПОЛЬЗОВАТЕЛИ, ДЛЯ КОТОРЫХ ПРОВЕРЯЕМ COVERAGE
# ============================================================

# ============================================================
# 5. ПОЛЬЗОВАТЕЛИ, ДЛЯ КОТОРЫХ ПРОВЕРЯЕМ COVERAGE
# ============================================================

positive_known = positive_pairs[
    positive_pairs["user_id"].isin(user_encoder.classes_)
    & positive_pairs["item_id"].isin(item_encoder.classes_)
].copy()


positive_user_ids = (
    positive_known["user_id"]
    .unique()
)


positive_user_idx = (
    user_encoder.transform(
        positive_user_ids
    )
)


print(
    "Количество известных пользователей:",
    len(positive_user_ids)
)

print(
    "Количество positive pairs, доступных ALS:",
    len(positive_known)
)

# ============================================================
# 6. ЗАДАЁМ TOP-N ДЛЯ ПРОВЕРКИ
# ============================================================

top_n_grid = [
    100,
    200,
    500,
    1000,
    2000,
    3000,
    5000
]


# Максимальное N должно быть не меньше максимума из grid
MAX_N = max(top_n_grid)


# На всякий случай ограничиваем количеством известных товаров
MAX_N = min(
    MAX_N,
    len(item_encoder.classes_)
)


print(
    "Максимальный TOP-N:",
    MAX_N
)


# ============================================================
# 7. ПОЛУЧАЕМ МАКСИМАЛЬНЫЙ TOP-N ОДИН РАЗ
# ============================================================

item_ids_test, scores_test = model_test.recommend(
    userid=positive_user_idx,
    user_items=user_item_matrix_test[
        positive_user_idx
    ],
    N=MAX_N,
    filter_already_liked_items=False
)


print(
    "Размер рекомендаций:",
    item_ids_test.shape
)


# ============================================================
# 8. ПРОВЕРЯЕМ РАЗНЫЕ РАЗМЕРЫ TOP-N
# ============================================================

results_n = []


for n in top_n_grid:

    # Реально доступное количество рекомендаций
    actual_n = min(
        n,
        item_ids_test.shape[1]
    )

    # Берём первые actual_n рекомендаций
    item_ids_n = item_ids_test[
        :,
        :actual_n
    ]


    # Возвращаем реальные item_id
    original_items = (
        item_encoder.classes_[
            item_ids_n.reshape(-1)
        ]
    )


    # Формируем DataFrame рекомендаций
    recommendations_n = pd.DataFrame({
        "user_id": np.repeat(
            positive_user_ids,
            actual_n
        ),

        "item_id": original_items
    })


    # Проверяем попадание positive pairs
    positive_found = positive_known.merge(
        recommendations_n,
        on=[
            "user_id",
            "item_id"
        ],
        how="inner"
    )


    coverage = (
    len(positive_found)
    / len(positive_known)
)


    results_n.append({
        "requested_top_n": n,
        "actual_top_n": actual_n,
        "positive_found": len(
            positive_found
        ),
        "positive_total": len(positive_known),
        "coverage": coverage
    })


    print(
        f"TOP-{n} "
        f"(фактически {actual_n}): "
        f"{len(positive_found)} / "
        f"{len(positive_known)} "
        f"= {coverage:.4f}"
    )


# ============================================================
# 9. ИТОГОВАЯ ТАБЛИЦА
# ============================================================

results_n_df = pd.DataFrame(
    results_n
)


results_n_df

Матрица: (1280329, 225427)
Количество взаимодействий: 1950423


  0%|          | 0/15 [00:00<?, ?it/s]

Количество известных пользователей: 303
Количество positive pairs, доступных ALS: 714
Максимальный TOP-N: 5000
Размер рекомендаций: (303, 5000)
TOP-100 (фактически 100): 86 / 714 = 0.1204
TOP-200 (фактически 200): 109 / 714 = 0.1527
TOP-500 (фактически 500): 154 / 714 = 0.2157
TOP-1000 (фактически 1000): 194 / 714 = 0.2717
TOP-2000 (фактически 2000): 242 / 714 = 0.3389
TOP-3000 (фактически 3000): 271 / 714 = 0.3796
TOP-5000 (фактически 5000): 316 / 714 = 0.4426


,requested_top_n,actual_top_n,positive_found,positive_total,coverage
0,100,100,86,714,0.120448
1,200,200,109,714,0.152661
2,500,500,154,714,0.215686
3,1000,1000,194,714,0.271709
4,2000,2000,242,714,0.338936
5,3000,3000,271,714,0.379552
6,5000,5000,316,714,0.442577


In [78]:
%reset -f

import gc
gc.collect()

0